# Authentication

**Objective:** Verify credentials safely and issue short-lived access tokens.

## Simple version

In [ ]:
import jwt


secret = "demo-secret-with-at-least-32-characters"
token = jwt.encode({"sub": "user-1"}, secret, algorithm="HS256")
payload = jwt.decode(token, secret, algorithms=["HS256"])

print(payload["sub"])

## Polished version

In [ ]:
import hashlib
import hmac
import secrets
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone

import jwt


class PasswordHasher:
    iterations = 100_000

    def hash(self, password: str) -> str:
        salt = secrets.token_bytes(16)
        digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, self.iterations)
        return f"{salt.hex()}:{digest.hex()}"

    def verify(self, password: str, encoded: str) -> bool:
        salt_hex, expected_hex = encoded.split(":", maxsplit=1)
        actual = hashlib.pbkdf2_hmac(
            "sha256",
            password.encode(),
            bytes.fromhex(salt_hex),
            self.iterations,
        )
        return hmac.compare_digest(actual.hex(), expected_hex)


@dataclass(frozen=True)
class User:
    id: int
    email: str
    password_hash: str


class AuthService:
    def __init__(self, secret: str) -> None:
        self.secret = secret
        self.passwords = PasswordHasher()
        self.users: dict[str, User] = {}

    def register(self, email: str, password: str) -> User:
        email = email.strip().lower()
        if email in self.users:
            raise ValueError("email already registered")
        user = User(len(self.users) + 1, email, self.passwords.hash(password))
        self.users[email] = user
        return user

    def login(self, email: str, password: str) -> str:
        user = self.users[email]
        if not self.passwords.verify(password, user.password_hash):
            raise ValueError("invalid credentials")
        expires = datetime.now(timezone.utc) + timedelta(minutes=15)
        return jwt.encode({"sub": str(user.id), "exp": expires}, self.secret, "HS256")


auth = AuthService("demo-secret-with-at-least-32-characters")
auth.register("ada@example.com", "password123")
print(auth.login("ada@example.com", "password123")[:24] + "...")